# ORIE 5132 — Choice Modeling, Assortment Optimization, Pricing, and AI Agents

This notebook contains Problems 1 through 7.


## Setup

In [29]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import logsumexp
from ortools.linear_solver import pywraplp

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [30]:
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "data.csv").exists()),
    Path.cwd(),
)
DATA_DIR = PROJECT_ROOT / "data"
FULL_DATA = DATA_DIR / "data.csv"
ASSORTMENT_FILES = [DATA_DIR / f"data{i}.csv" for i in range(1, 5)]

FEATURES = [
    "prop_starrating",
    "prop_review_score",
    "prop_brand_bool",
    "prop_location_score",
    "prop_accesibility_score",
    "prop_log_historical_price",
    "price_usd",
    "promotion_flag",
]

PARAM_NAMES = ["intercept"] + FEATURES
PRICE_COL = "price_usd"


### Load and validate the data

In [31]:
df = pd.read_csv(FULL_DATA)

required = {
    "srch_id",
    "srch_booking_window",
    "srch_adults_count",
    "srch_children_count",
    "srch_room_count",
    "srch_saturday_night_bool",
    "booking_bool",
    *FEATURES,
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"data.csv is missing required columns: {missing}")

df = df.dropna(subset=list(required)).copy()

print(f"Rows: {len(df):,}")
print(f"Unique search queries: {df['srch_id'].nunique():,}")
df.head(10)


Rows: 153,009
Unique search queries: 8,354


,srch_id,prop_starrating,prop_review_score,prop_brand_bool,prop_location_score,prop_accesibility_score,prop_log_historical_price,price_usd,promotion_flag,srch_booking_window,srch_adults_count,srch_children_count,srch_room_count,srch_saturday_night_bool,booking_bool
0,1,4,3,1,2,0,5,140,0,0,4,0,1,1,0
1,1,3,5,1,2,0,5,211,0,0,4,0,1,1,0
2,1,4,4,1,3,0,5,150,0,0,4,0,1,1,0
3,1,4,3,1,3,0,5,144,0,0,4,0,1,1,0
4,1,4,4,1,2,0,5,191,0,0,4,0,1,1,0
5,1,4,5,1,3,0,5,195,0,0,4,0,1,1,0
6,1,4,3,1,3,0,5,115,0,0,4,0,1,1,0
7,1,4,4,1,3,0,5,281,0,0,4,0,1,1,0
8,1,2,3,1,3,0,4,128,0,0,4,0,1,1,0
9,1,2,3,1,2,0,4,101,0,0,4,0,1,1,1


## Problem 1 — MNL Model

Customers are assumed to choose according to an MNL model. For hotel $j$ with features $x_{ji}$ and customer feature sensitivities $\beta_i$:

$$u_j = \beta_0 + \sum_{i=1}^8 \beta_i x_{ji}, \quad v_j = e^{u_j}, \quad \mathbb{P}(j \mid S) = \frac{v_j}{1 + \sum_{p \in S} v_p}$$

The log-likelihood for one search query is

$$\ell_q(\beta) = \sum_{j \in S_q} y_j u_j - \log\!\left(1 + \sum_{j \in S_q} e^{u_j}\right)$$

where $y_j$ is the booking indicator. We estimate $\beta_0,\dots,\beta_8$ by maximum likelihood.

In [32]:
X = df[FEATURES].to_numpy(dtype=float)
y = df["booking_bool"].to_numpy(dtype=float)
groups = df["srch_id"].to_numpy()

In [33]:
def compute_utilities(beta, X):
    return beta[0] + X @ beta[1:]


def prepare_groups(groups):
    """Map raw srch_id values to dense indices 0..n_groups-1 for fast bincount."""
    _, group_index = np.unique(groups, return_inverse=True)
    return group_index


def neg_log_likelihood(beta, X, y, group_index):
    u = compute_utilities(beta, X)

    booked_utility = y @ u

    # vectorized: log(1 + sum exp(u_j)) summed over groups
    sum_exp_by_group = np.bincount(group_index, weights=np.exp(u))
    log_denom = np.log1p(sum_exp_by_group).sum()

    return -(booked_utility - log_denom)

In [34]:
group_index = prepare_groups(groups)
beta_init = np.zeros(len(PARAM_NAMES))

result = minimize(
    neg_log_likelihood,
    beta_init,
    args=(X, y, group_index),
    method="L-BFGS-B",
    options={"maxiter": 1000},
)

beta_hat = result.x

print(f"Converged: {result.success}; negative log-likelihood: {result.fun:.4f}")
print()
for name, val in zip(PARAM_NAMES, beta_hat):
    print(f"{name:>30s}: {val:+.4f}")

Converged: True; negative log-likelihood: 20611.3618

                     intercept: -2.8305
               prop_starrating: +0.4771
             prop_review_score: +0.1214
               prop_brand_bool: +0.2346
           prop_location_score: +0.0173
       prop_accesibility_score: +0.5521
     prop_log_historical_price: -0.0367
                     price_usd: -0.0073
                promotion_flag: +0.4488


### Commentary on coefficients

All signs are intuitively correct:

- **prop_starrating** (+0.48) and **promotion_flag** (+0.46) are the strongest positive drivers — higher star rating and promotions both boost booking probability.
- **prop_brand_bool** (+0.23) and **prop_review_score** (+0.12) are also positive, consistent with customers preferring branded hotels and higher reviews.
- **prop_accesibility_score** (+0.57) is surprisingly the largest positive coefficient. It is likely capturing something correlated with desirability beyond pure accessibility.
- **prop_location_score** (+0.02) is nearly zero, possibly due to limited variance in the column or collinearity with other features.
- **price_usd** (-0.0073) is correctly negative. Per dollar this looks small, but prices range $40–$600, so a $100 increase reduces utility by ~0.73 — comparable to losing 1.5 star rating points.
- **prop_log_historical_price** (-0.035) is also negative; its correlation with `price_usd` likely splits the price-sensitivity effect across the two coefficients.

## Problem 2 — Assortment Optimization under MNL

For each of `data1.csv`, `data2.csv`, `data3.csv`, `data4.csv`, find the optimal subset $S$ of hotels to display:

$$R(S) = \frac{\sum_{j \in S} p_j v_j}{1 + \sum_{j \in S} v_j}$$

By the **revenue-ordered property** of MNL, the optimal $S$ is always a prefix of the price-sorted list. Walk down the sorted list and add hotel $k$ if and only if $p_k > R(\text{current})$. Once a hotel fails this test, all remaining hotels also fail (they have lower prices), so we can stop.

In [35]:
def optimal_assortment(df_hotels, beta, features=FEATURES):
    df_local = df_hotels.copy()

    u = compute_utilities(beta, df_local[features].to_numpy(dtype=float))
    df_local["v"] = np.exp(u)

    df_local = df_local.sort_values("price_usd", ascending=False).reset_index(drop=True)

    num = 0.0  # sum of p_j * v_j
    den = 1.0  # 1 + sum of v_j (the 1 represents the no-purchase option)
    chosen_idx = []

    for i, row in df_local.iterrows():
        p = row["price_usd"]
        v = row["v"]
        R_current = num / den

        if p > R_current:
            num += p * v
            den += v
            chosen_idx.append(i)
        else:
            break

    S = df_local.loc[chosen_idx]
    R_opt = num / den
    return S, R_opt

In [36]:
for i in range(1, 5):
    df_i = pd.read_csv(DATA_DIR / f"data{i}.csv")
    S, R_opt = optimal_assortment(df_i, beta_hat)
    print(f"--- data{i}.csv ---")
    print(f"Optimal assortment size: {len(S)} hotels")
    print(f"Expected revenue: ${R_opt:.2f}")
    print(S[["price_usd", "v"]].to_string())
    print()


--- data1.csv ---
Optimal assortment size: 18 hotels
Expected revenue: $107.37
    price_usd        v
0         158 0.200595
1         156 0.248112
2         154 0.153874
3         150 0.140341
4         147 0.381461
5         147 0.381461
6         145 0.167260
7         144 0.168492
8         140 0.173514
9         131 0.167043
10        131 0.164172
11        125 0.306795
12        121 0.128123
13        121 0.156478
14        120 0.200960
15        118 0.211127
16        112 0.355538
17        108 0.097068

--- data2.csv ---
Optimal assortment size: 10 hotels
Expected revenue: $131.14
   price_usd        v
0        233 0.293072
1        212 0.535605
2        206 0.357332
3        161 0.308572
4        147 0.435839
5        146 0.344499
6        143 0.352171
7        137 0.593056
8        134 0.412802
9        133 0.610733

--- data3.csv ---
Optimal assortment size: 18 hotels
Expected revenue: $121.09
    price_usd        v
0         603 0.003420
1         281 0.091044
2         211

### Discussion

Across the four datasets the optimal assortments range from 10 to 18 hotels and expected revenues from \$97 to \$131.

- **data2** has the highest expected revenue (\$130.99) with only 10 hotels — its hotels combine relatively high prices with healthy preference weights.
- **data3** is interesting: a \$603 outlier with $v = 0.003$ is included anyway, because its price is so far above the running expected revenue that it clears the threshold despite a tiny choice probability.
- **data4** is weakest at \$97.25 — prices cap at \$195 and the assortment is dominated by hotels around \$100, dragging the average down.

## Problem 3 — Pricing under MNL

Display all hotels in `data{i}.csv` and choose new prices to maximize expected revenue. Under MNL with a shared price coefficient $\beta_p$, the first-order condition gives

$$p_j^* = R^* + \frac{1}{|\beta_p|}, \quad \forall j$$

so **all hotels share the same optimal price**. We reduce to a 1D maximization over a scalar $p$:

$$R(p) = \frac{p \sum_j e^{a_j + \beta_p p}}{1 + \sum_j e^{a_j + \beta_p p}}$$

where $a_j = \beta_0 + \sum_{i \ne \text{price}} \beta_i x_{ji}$ is the intrinsic utility (everything except the price contribution).

In [37]:
def optimal_pricing(df_hotels, beta, features=FEATURES):
    X_h = df_hotels[features].to_numpy(dtype=float)

    price_idx = features.index("price_usd")
    beta_price = beta[price_idx + 1]   # +1 since beta[0] is the intercept

    # intrinsic utility: zero out the price contribution
    X_no_price = X_h.copy()
    X_no_price[:, price_idx] = 0.0
    a = beta[0] + X_no_price @ beta[1:]

    def neg_revenue(p):
        v = np.exp(a + beta_price * p)
        return -p * v.sum() / (1 + v.sum())

    result = minimize_scalar(neg_revenue, bounds=(1.0, 5000.0), method="bounded")

    p_star = result.x
    R_star = -result.fun
    markup = 1.0 / abs(beta_price)
    return p_star, R_star, markup

In [38]:
for i in range(1, 5):
    df_i = pd.read_csv(DATA_DIR / f"data{i}.csv")
    p_star, R_star, markup = optimal_pricing(df_i, beta_hat)
    print(f"--- data{i}.csv ---")
    print(f"  Optimal price (all hotels):    ${p_star:.2f}")
    print(f"  Expected revenue:              ${R_star:.2f}")
    print(f"  Theoretical markup 1/|beta_p|: ${markup:.2f}")
    print(f"  p* - R* (should match markup): ${p_star - R_star:.2f}")
    print()


--- data1.csv ---
  Optimal price (all hotels):    $313.79
  Expected revenue:              $177.59
  Theoretical markup 1/|beta_p|: $136.20
  p* - R* (should match markup): $136.20

--- data2.csv ---
  Optimal price (all hotels):    $384.66
  Expected revenue:              $248.47
  Theoretical markup 1/|beta_p|: $136.20
  p* - R* (should match markup): $136.20

--- data3.csv ---
  Optimal price (all hotels):    $312.50
  Expected revenue:              $176.31
  Theoretical markup 1/|beta_p|: $136.20
  p* - R* (should match markup): $136.20

--- data4.csv ---
  Optimal price (all hotels):    $349.89
  Expected revenue:              $213.70
  Theoretical markup 1/|beta_p|: $136.20
  p* - R* (should match markup): $136.20



### Discussion

The FOC sanity check holds across all four datasets: $p^* - R^* = 1/|\beta_p| = \$136.56$ exactly.

- **data2** has the highest optimal price (\$384.87) and revenue (\$248.30) — its hotels have the strongest intrinsic appeal, so customers tolerate the highest pricing.
- **data1** and **data3** have similar p* (~\$313) and R* (~\$177), suggesting comparable intrinsic appeal on average even though data3 contained the \$603 outlier in P2.
- **data4** lands in between (\$350 / \$214).

The optimal prices are much higher than typical market prices in P2's assortments because the firm is now setting prices freely rather than working with the prices in the dataset.

## Problem 4 — Mixture of MNL for Early vs. Late Reservations

Customer types based on `srch_booking_window` (days):

- **Type 1 / early**: `srch_booking_window >= 7`
- **Type 2 / late**:  `srch_booking_window < 7`

For type $k$:

$$u_{jk} = \beta_{0k} + \sum_i \beta_{ik} x_{ji}, \quad v_{jk} = e^{u_{jk}}, \quad \mathbb{P}(j \mid S) = \sum_{k=1}^2 \theta_k \frac{v_{jk}}{1 + \sum_{p \in S} v_{pk}}$$

We estimate $\theta_1, \theta_2$ by counting unique queries of each type, and fit one MNL per type by MLE on its rows.

### Define early and late customer types

Type shares are computed by **unique search queries** so they are not biased by queries that displayed more hotels.

In [39]:
def query_type_table(df: pd.DataFrame) -> pd.DataFrame:
    per_query = (
        df[["srch_id", "srch_booking_window"]]
        .drop_duplicates(subset="srch_id")
        .copy()
    )
    per_query["customer_type"] = np.where(
        per_query["srch_booking_window"] >= 7,
        "type_1_early",
        "type_2_late",
    )
    return per_query

per_query = query_type_table(df)
per_query.head(10)

,srch_id,srch_booking_window,customer_type
0,1,0,type_2_late
26,75,30,type_1_early
59,101,74,type_1_early
91,218,0,type_2_late
96,236,15,type_1_early
121,325,12,type_1_early
135,419,53,type_1_early
148,1069,21,type_1_early
152,1297,1,type_2_late
168,1555,35,type_1_early


In [40]:
n_queries = per_query["srch_id"].nunique()
theta_1 = per_query["customer_type"].eq("type_1_early").sum() / n_queries
theta_2 = per_query["customer_type"].eq("type_2_late").sum() / n_queries

theta_table = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "definition": ["srch_booking_window >= 7", "srch_booking_window < 7"],
    "theta": [theta_1, theta_2],
    "searches": [
        per_query["customer_type"].eq("type_1_early").sum(),
        per_query["customer_type"].eq("type_2_late").sum(),
    ],
})

print(f"theta_1 + theta_2 = {theta_1 + theta_2:.6f}")
theta_table

theta_1 + theta_2 = 1.000000


,type,definition,theta,searches
0,type_1_early,srch_booking_window >= 7,0.543093,4537
1,type_2_late,srch_booking_window < 7,0.456907,3817


### Attach customer type to row-level hotel data, then estimate one MNL per type

In [41]:
df = df.drop(columns=["customer_type"], errors="ignore").merge(
    per_query[["srch_id", "customer_type"]],
    on="srch_id",
    how="left",
)

df_early = df[df["customer_type"] == "type_1_early"].copy()
df_late = df[df["customer_type"] == "type_2_late"].copy()

summary_by_type = pd.DataFrame({
    "type": ["type_1_early", "type_2_late"],
    "searches": [df_early["srch_id"].nunique(), df_late["srch_id"].nunique()],
    "rows": [len(df_early), len(df_late)],
    "bookings": [df_early["booking_bool"].sum(), df_late["booking_bool"].sum()],
})
summary_by_type


,type,searches,rows,bookings
0,type_1_early,4537,83863,3047
1,type_2_late,3817,69146,2801


In [42]:
def estimate_mnl(df_type, label, target_col="booking_bool"):
    X_t = df_type[FEATURES].to_numpy(dtype=float)
    y_t = df_type[target_col].to_numpy(dtype=float)
    g_t = prepare_groups(df_type["srch_id"].to_numpy())

    beta_init = np.zeros(len(PARAM_NAMES))

    res = minimize(
        neg_log_likelihood,
        beta_init,
        args=(X_t, y_t, g_t),
        method="L-BFGS-B",
        options={"maxiter": 1000},
    )

    if not res.success:
        print(f"WARNING: {label} did not converge: {res.message}")
    return res.x, res

beta_early, result_early = estimate_mnl(df_early, "type 1 early")
beta_late,  result_late  = estimate_mnl(df_late,  "type 2 late")

print(f"Early: success={result_early.success}; neg-log-lik={result_early.fun:.4f}")
print(f"Late:  success={result_late.success};  neg-log-lik={result_late.fun:.4f}")


Early: success=True; neg-log-lik=11052.5871
Late:  success=True;  neg-log-lik=9505.4483


In [43]:
beta_table = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_type_1_early": beta_early,
    "beta_type_2_late": beta_late,
    "early_minus_late": beta_early - beta_late,
    "abs_difference": np.abs(beta_early - beta_late),
})

beta_table.sort_values("abs_difference", ascending=False)

,parameter,beta_type_1_early,beta_type_2_late,early_minus_late,abs_difference
5,prop_accesibility_score,0.749543,0.329183,0.420360,0.420360
0,intercept,-2.974746,-2.708656,-0.266090,0.266090
8,promotion_flag,0.382883,0.552010,-0.169127,0.169127
1,prop_starrating,0.442256,0.539868,-0.097612,0.097612
4,prop_location_score,-0.015951,0.064270,-0.080221,0.080221
3,prop_brand_bool,0.208882,0.256062,-0.047180,0.047180
6,prop_log_historical_price,-0.053046,-0.017721,-0.035325,0.035325
2,prop_review_score,0.137347,0.105208,0.032139,0.032139
7,price_usd,-0.005896,-0.009338,0.003442,0.003442


### Commentary

- **Mixture weights**: $\theta_1 \approx 0.543$, $\theta_2 \approx 0.457$. Slightly more early bookers than late bookers.
- **Largest difference**: `prop_accesibility_score` differs sharply between the two types. Early customers weight accessibility much more heavily.
- **Promotions**: late customers respond more strongly to promotion flags and star rating than early ones.
- All other coefficients differ by less than 0.1 in absolute terms.

## Problem 5 — Early vs. Late Reservations (Assortment Optimization)

For each of `data1.csv`, `data2.csv`, `data3.csv`, `data4.csv`, solve three assortment problems:

- $S$: optimal assortment when the arriving customer's type is **unknown** (mixture model).
- $S_1$: optimal assortment for a known **type 1** customer.
- $S_2$: optimal assortment for a known **type 2** customer.

Then compare:

- Revenue of $S$ vs. $S_1$ under the type 1 MNL.
- Revenue of $S$ vs. $S_2$ under the type 2 MNL.

For known types, the revenue-ordered property holds and we use a greedy prefix scan. For the unknown-type mixture problem, the property breaks down so we solve a mixed-integer linear program.

### MILP formulation

The mixture revenue is

$$R_{\text{mix}}(x) = \theta_1 \frac{\sum_j x_j r_j v_{j1}}{1 + \sum_j x_j v_{j1}} + \theta_2 \frac{\sum_j x_j r_j v_{j2}}{1 + \sum_j x_j v_{j2}}$$

Linearize with the substitutions $z_k = 1 / (1 + \sum_j v_{jk} x_j)$ and $y_{jk} = x_j z_k$. Then the denominator relationship $z_k + \sum_j v_{jk} y_{jk} = 1$ enforces the inverse, and the revenue per type becomes $\sum_j r_j v_{jk} y_{jk}$. Linearize $y_{jk} = x_j z_k$ via:

$$y_{jk} \le z_k, \quad y_{jk} \le x_j, \quad y_{jk} \ge z_k - (1 - x_j)$$

with $x_j \in \{0,1\}$, $z_k \in [0,1]$, $y_{jk} \in [0,1]$. Maximize $\sum_k \theta_k \sum_j r_j v_{jk} y_{jk}$.

In [44]:
def compute_preference_weights(df_hotels, beta):
    X_h = df_hotels[FEATURES].to_numpy(dtype=float)
    u = compute_utilities(beta, X_h)
    u = np.clip(u, -50, 50)   # prevent overflow in exp
    return np.exp(u)


def expected_revenue_type(x, revenue, v):
    x = np.asarray(x)
    return np.sum(x * revenue * v) / (1 + np.sum(x * v))


def expected_revenue_mixture(x, revenue, v1, v2, theta1, theta2):
    return (
        theta1 * expected_revenue_type(x, revenue, v1)
        + theta2 * expected_revenue_type(x, revenue, v2)
    )


In [45]:
def optimal_assortment_single_type(df_hotels, v, price_col=PRICE_COL):
    df_local = df_hotels.copy()
    df_local["_original_index"] = np.arange(len(df_local))
    df_local["_v"] = v

    df_local = df_local.sort_values(price_col, ascending=False).reset_index(drop=True)

    num = 0.0
    den = 1.0
    chosen_original_indices = []

    for _, row in df_local.iterrows():
        price = row[price_col]
        weight = row["_v"]
        if price > num / den:
            num += price * weight
            den += weight
            chosen_original_indices.append(int(row["_original_index"]))
        else:
            break

    selected = np.zeros(len(df_hotels), dtype=int)
    selected[chosen_original_indices] = 1
    return selected, num / den

In [46]:
def solve_assortment_ortools(revenue, v_by_type, theta_by_type, time_limit_seconds=300):
    revenue = np.asarray(revenue, dtype=float)
    v_by_type = [np.asarray(v, dtype=float) for v in v_by_type]

    n = len(revenue)
    K = len(v_by_type)

    solver = pywraplp.Solver.CreateSolver("CBC")
    if solver is None:
        raise RuntimeError("CBC solver is not available. Try reinstalling OR-Tools.")

    solver.SetTimeLimit(int(time_limit_seconds * 1000))

    hotels = range(n)
    types = range(K)

    x = {j: solver.BoolVar(f"x_{j}") for j in hotels}
    z = {k: solver.NumVar(0.0, 1.0, f"z_{k}") for k in types}
    y = {(j, k): solver.NumVar(0.0, 1.0, f"y_{j}_{k}") for j in hotels for k in types}

    for k in types:
        solver.Add(z[k] + solver.Sum(v_by_type[k][j] * y[j, k] for j in hotels) == 1)

    for j in hotels:
        for k in types:
            solver.Add(y[j, k] <= z[k])
            solver.Add(y[j, k] <= x[j])
            solver.Add(y[j, k] >= z[k] - (1 - x[j]))

    objective = solver.Sum(
        theta_by_type[k] * revenue[j] * v_by_type[k][j] * y[j, k]
        for k in types for j in hotels
    )
    solver.Maximize(objective)

    status = solver.Solve()

    if status not in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
        print(f"WARNING: solver did not find an optimal/feasible solution. Status: {status}")

    selected = np.array([int(round(x[j].solution_value())) for j in hotels])
    return selected, solver.Objective().Value(), status

In [47]:
def interpret_status(status):
    if status == pywraplp.Solver.OPTIMAL:    return "OPTIMAL"
    if status == pywraplp.Solver.FEASIBLE:   return "FEASIBLE"
    if status == pywraplp.Solver.INFEASIBLE: return "INFEASIBLE"
    if status == pywraplp.Solver.UNBOUNDED:  return "UNBOUNDED"
    return f"OTHER ({status})"


def solve_mixture_assortment_for_dataset(file_path, beta_type1, beta_type2, theta_type1, theta_type2):
    df_hotels = pd.read_csv(file_path).copy()
    revenue = df_hotels[PRICE_COL].to_numpy(dtype=float)

    v1 = compute_preference_weights(df_hotels, beta_type1)
    v2 = compute_preference_weights(df_hotels, beta_type2)

    # S: unknown customer type — solve MILP
    S, mix_obj, status_mix = solve_assortment_ortools(
        revenue=revenue,
        v_by_type=[v1, v2],
        theta_by_type=[theta_type1, theta_type2],
    )

    # S1, S2: known type — revenue-ordered greedy is exact
    S1, _ = optimal_assortment_single_type(df_hotels, v1)
    S2, _ = optimal_assortment_single_type(df_hotels, v2)

    rev_S_under_type1  = expected_revenue_type(S,  revenue, v1)
    rev_S1_under_type1 = expected_revenue_type(S1, revenue, v1)
    rev_S_under_type2  = expected_revenue_type(S,  revenue, v2)
    rev_S2_under_type2 = expected_revenue_type(S2, revenue, v2)

    summary = {
        "dataset": Path(file_path).name,
        "n_hotels": len(df_hotels),
        "S_size": int(S.sum()),
        "S1_size": int(S1.sum()),
        "S2_size": int(S2.sum()),
        "mixture_revenue_of_S": mix_obj,
        "revenue_S_under_type1": rev_S_under_type1,
        "revenue_S1_under_type1": rev_S1_under_type1,
        "type1_gain_from_personalization": rev_S1_under_type1 - rev_S_under_type1,
        "revenue_S_under_type2": rev_S_under_type2,
        "revenue_S2_under_type2": rev_S2_under_type2,
        "type2_gain_from_personalization": rev_S2_under_type2 - rev_S_under_type2,
        "status_mix": interpret_status(status_mix),
    }

    selected_hotels = {
        "S_unknown_type": df_hotels.loc[S == 1].copy(),
        "S1_type1":       df_hotels.loc[S1 == 1].copy(),
        "S2_type2":       df_hotels.loc[S2 == 1].copy(),
    }
    return summary, selected_hotels


In [48]:
all_results = []
all_selected_hotels = {}

for file_path in ASSORTMENT_FILES:
    print(f"Solving {file_path.name}...")
    result_summary, selected_hotels = solve_mixture_assortment_for_dataset(
        file_path,
        beta_early,
        beta_late,
        theta_1,
        theta_2,
    )
    all_results.append(result_summary)
    all_selected_hotels[file_path] = selected_hotels

results_df = pd.DataFrame(all_results)
results_df


Solving data1.csv...
Solving data2.csv...
Solving data3.csv...
Solving data4.csv...


,dataset,n_hotels,S_size,S1_size,S2_size,mixture_revenue_of_S,revenue_S_under_type1,revenue_S1_under_type1,type1_gain_from_personalization,revenue_S_under_type2,revenue_S2_under_type2,type2_gain_from_personalization,status_mix
0,data1.csv,27,18,21,16,107.302190,103.251968,103.470005,0.218037,112.116405,112.217873,0.101468,OPTIMAL
1,data2.csv,29,10,10,7,131.255081,126.923107,126.923107,0.000000,136.404193,137.294880,0.890686,OPTIMAL
2,data3.csv,26,18,19,17,120.990462,117.436396,117.519089,0.082693,125.214931,125.333514,0.118583,OPTIMAL
3,data4.csv,27,11,11,10,97.378416,94.695474,94.695474,0.000000,100.567442,100.628550,0.061108,OPTIMAL


In [49]:
for file_path in ASSORTMENT_FILES:
    print(f"\n--- {file_path.name} ---")
    print("S  (unknown type):", list(all_selected_hotels[file_path]["S_unknown_type"].index))
    print("S1 (type 1):     ", list(all_selected_hotels[file_path]["S1_type1"].index))
    print("S2 (type 2):     ", list(all_selected_hotels[file_path]["S2_type2"].index))



--- data1.csv ---
S  (unknown type): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
S1 (type 1):      [0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26]
S2 (type 2):      [0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24]

--- data2.csv ---
S  (unknown type): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
S1 (type 1):      [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
S2 (type 2):      [0, 1, 6, 7, 8, 10, 21]

--- data3.csv ---
S  (unknown type): [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
S1 (type 1):      [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24, 25]
S2 (type 2):      [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 19, 23, 24]

--- data4.csv ---
S  (unknown type): [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
S1 (type 1):      [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
S2 (type 2):      [3, 4, 6, 10, 15, 18, 19, 20, 21, 26]


### Discussion

For each dataset we compared the mixture-optimal assortment $S$ against the type-specific assortments $S_1$ and $S_2$:

- Under type 1: $S_1$ performs as well as or slightly better than $S$. Gains are small (on the order of $0$–$\$0.30$).
- Under type 2: $S_2$ consistently performs better than $S$ — late customers benefit more from personalization.

**Takeaway**: knowing the customer type lets us slightly improve revenue by tailoring the assortment. The gains are not large, which makes sense given that the early- and late-customer MNLs are not dramatically different (most coefficients differ by less than 0.1). The mixture solution is a reasonable compromise when the type is unknown.

## Problem 6 — Mixture MNL for Saturday vs. Non-Saturday Stays


In [50]:
PROBLEM6_TYPE_COL = "problem6_type"

problem6_df = df.dropna(subset=["srch_saturday_night_bool", "booking_bool", *FEATURES]).copy()
problem6_df[PROBLEM6_TYPE_COL] = np.where(
    problem6_df["srch_saturday_night_bool"].astype(int).eq(1),
    "type_1_saturday_night",
    "type_2_non_saturday_night",
)

problem6_query_types = problem6_df[["srch_id", PROBLEM6_TYPE_COL]].drop_duplicates()
problem6_theta = problem6_query_types[PROBLEM6_TYPE_COL].value_counts(normalize=True).sort_index()

problem6_theta_table = pd.DataFrame({
    "type": problem6_theta.index,
    "definition": ["srch_saturday_night_bool == 1", "srch_saturday_night_bool == 0"],
    "theta": problem6_theta.to_numpy(),
    "searches": problem6_query_types[PROBLEM6_TYPE_COL].value_counts().sort_index().to_numpy(),
})
problem6_theta_table


,type,definition,theta,searches
0,type_1_saturday_night,srch_saturday_night_bool == 1,0.576610,4817
1,type_2_non_saturday_night,srch_saturday_night_bool == 0,0.423390,3537


In [51]:
df_saturday = problem6_df[problem6_df[PROBLEM6_TYPE_COL] == "type_1_saturday_night"].copy()
df_non_saturday = problem6_df[problem6_df[PROBLEM6_TYPE_COL] == "type_2_non_saturday_night"].copy()

beta_saturday, result_saturday = estimate_mnl(df_saturday, "type 1 saturday night")
beta_non_saturday, result_non_saturday = estimate_mnl(df_non_saturday, "type 2 non-saturday night")

problem6_beta_table = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_saturday_night": beta_saturday,
    "beta_non_saturday_night": beta_non_saturday,
})
problem6_beta_table["difference"] = problem6_beta_table["beta_saturday_night"] - problem6_beta_table["beta_non_saturday_night"]
problem6_beta_table


,parameter,beta_saturday_night,beta_non_saturday_night,difference
0,intercept,-2.817626,-2.854266,0.036640
1,prop_starrating,0.421951,0.551201,-0.129250
2,prop_review_score,0.121088,0.120590,0.000498
3,prop_brand_bool,0.223191,0.246434,-0.023242
4,prop_location_score,0.020046,0.011610,0.008437
5,prop_accesibility_score,0.453535,0.678083,-0.224548
6,prop_log_historical_price,-0.033386,-0.038584,0.005198
7,price_usd,-0.006368,-0.008620,0.002252
8,promotion_flag,0.475542,0.419441,0.056101


In [52]:
problem6_rows = []
problem6_selected_hotels = {}

theta_saturday = problem6_theta.loc["type_1_saturday_night"]
theta_non_saturday = problem6_theta.loc["type_2_non_saturday_night"]

for file_path in ASSORTMENT_FILES:
    print(f"Solving {file_path.name}...")
    summary, selected_hotels = solve_mixture_assortment_for_dataset(
        file_path,
        beta_saturday,
        beta_non_saturday,
        theta_saturday,
        theta_non_saturday,
    )
    problem6_rows.append(summary)
    problem6_selected_hotels[file_path] = selected_hotels

problem6_results_df = pd.DataFrame(problem6_rows)
problem6_results_df


Solving data1.csv...
Solving data2.csv...
Solving data3.csv...
Solving data4.csv...


,dataset,n_hotels,S_size,S1_size,S2_size,mixture_revenue_of_S,revenue_S_under_type1,revenue_S1_under_type1,type1_gain_from_personalization,revenue_S_under_type2,revenue_S2_under_type2,type2_gain_from_personalization,status_mix
0,data1.csv,27,18,20,17,107.357275,106.809907,106.819631,0.009724,108.102729,108.104701,0.001973,OPTIMAL
1,data2.csv,29,10,10,10,130.896699,129.871681,129.871681,0.000000,132.292659,132.292659,0.000000,OPTIMAL
2,data3.csv,26,18,18,18,121.189408,121.611210,121.611210,0.000000,120.614960,120.614960,0.000000,OPTIMAL
3,data4.csv,27,11,11,11,97.181494,96.507035,96.507035,0.000000,98.100031,98.100031,0.000000,OPTIMAL


## Problem 7 — AI Agents as Customers

We use **OpenAI GPT-4o mini via API** as our AI agent. The spec states agent choice does not affect grading; the goal is to compare agent behavior against the human-derived MNL. Because the hotel data is purely numeric (no descriptions or reviews to interpret), model size matters less here than in text-rich choice settings, so a smaller model is a reasonable budget choice.

We initially prototyped an Excel-batch upload workflow against the Claude.ai frontend, but the volume (~50 manual uploads) was too cumbersome and we opted for the API instead.

### No-purchase rate calibration

LLMs systematically over-purchase as choice simulators. The empirical no-purchase rate in `data.csv` is **30.0%** (2,506 of 8,354 queries). We target this rate via a stated 30% quota in the system prompt plus a running counter injected into each batch that nudges the model when it drifts off target.

### Enter your OpenAI API key

Run the cell below and paste your key when prompted. The key stays in memory only — never written to the notebook or disk. Alternatively, `export OPENAI_API_KEY=sk-...` in your shell before launching jupyter.

In [61]:
import os
import getpass

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key:
    api_key = getpass.getpass("OpenAI API key: ").strip()

if not api_key:
    raise ValueError("No OpenAI API key was provided.")

os.environ["OPENAI_API_KEY"] = api_key
print("OpenAI API key loaded into this notebook session.")


OpenAI API key loaded into this notebook session.


### API setup

In [62]:
import json as _json
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not api_key:
    raise ValueError("Run the API key cell above before creating the OpenAI client.")

client = OpenAI(api_key=api_key)

GPT_MODEL = "gpt-4o-mini"
GPT_BATCH_SIZE = 25         # queries per API call (~25-30k input tokens)
TARGET_NO_PURCHASE = 0.30   # 30.0% empirical rate from data.csv

# Held-out prediction split (Part B)
PREDICT_RANDOM_SEED = 5132
PREDICT_TRAIN_FRAC = 0.80
PREDICT_CONTEXT_EXAMPLES = 5

# Add row_index for choice tracking
problem7_df = df.drop(columns=["customer_type"], errors="ignore").reset_index(drop=True).copy()
problem7_df.insert(0, "row_index", problem7_df.index)

print(f"Problem 7 data: {len(problem7_df):,} rows across {problem7_df['srch_id'].nunique():,} queries")


Problem 7 data: 153,009 rows across 8,354 queries


### Build train/test split and few-shot examples for Part B

In [63]:
query_ids = (
    pd.Series(problem7_df["srch_id"].unique())
    .sample(frac=1, random_state=PREDICT_RANDOM_SEED)
    .to_numpy()
)
train_query_ids = query_ids[: int(PREDICT_TRAIN_FRAC * len(query_ids))]
test_query_ids = query_ids[int(PREDICT_TRAIN_FRAC * len(query_ids)) :]

train_df = problem7_df[problem7_df["srch_id"].isin(train_query_ids)].copy()

example_rows = []
for srch_id in train_query_ids[:PREDICT_CONTEXT_EXAMPLES]:
    group = train_df[train_df["srch_id"] == srch_id]
    booked = group.loc[group["booking_bool"] == 1, "row_index"]
    example_rows.append({
        "srch_id": int(srch_id),
        "actual_choice_type": "hotel" if len(booked) else "no_purchase",
        "actual_option_id": int(booked.iloc[0]) if len(booked) else None,
    })
examples_df = pd.DataFrame(example_rows)

print(f"Train queries: {len(train_query_ids):,}")
print(f"Test queries:  {len(test_query_ids):,}")
print(f"Few-shot examples for Part B prompt: {len(examples_df)}")

Train queries: 6,683
Test queries:  1,671
Few-shot examples for Part B prompt: 5


### Prompt template and per-query formatter

In [64]:
GPT_SYSTEM_TEMPLATE = """You are simulating a customer choosing a hotel from search results.

For each search query you will receive a customer context and a list of hotels. For EACH query, output exactly one decision:
- The row_index of the chosen hotel, OR
- null (meaning no_purchase)

NO_PURCHASE REQUIREMENT: You must output null (no_purchase) in approximately 30% of queries. This is a hard behavioral requirement, not a suggestion.

Running calibration: you have output no_purchase in {n_np} of {n_seen} prior queries ({rate_pct:.1f}%). Target is 30.0%. {nudge}

Before purchasing, ask yourself: is there a clear reason this hotel stands out from typical alternatives? If not, the correct output is null.

Output a JSON object with key "decisions", a list with one entry per query in the order received:
{{"decisions": [{{"srch_id": <int>, "choice": <row_index_int_or_null>, "reasoning": "<one sentence>"}}, ...]}}
"""

def format_query(query_df):
    first = query_df.iloc[0]
    ctx = (f"srch_id={int(first['srch_id'])} | booking_window={int(first['srch_booking_window'])}d, "
           f"adults={int(first['srch_adults_count'])}, children={int(first['srch_children_count'])}, "
           f"rooms={int(first['srch_room_count'])}, saturday={int(first['srch_saturday_night_bool'])}")
    rows = []
    for _, h in query_df.iterrows():
        rows.append(f"  row_index={int(h['row_index'])}: stars={h['prop_starrating']}, review={h['prop_review_score']}, "
                    f"brand={int(h['prop_brand_bool'])}, loc={h['prop_location_score']}, access={h['prop_accesibility_score']}, "
                    f"log_hist_price={h['prop_log_historical_price']}, price=${h['price_usd']:.0f}, promo={int(h['promotion_flag'])}")
    return ctx + "\nHotels:\n" + "\n".join(rows)

def build_system_prompt(n_no_purchase, n_seen):
    rate = n_no_purchase / n_seen if n_seen else 0.0
    if rate < TARGET_NO_PURCHASE - 0.02:
        nudge = "You are below target — lean toward null (no_purchase) for borderline queries."
    elif rate > TARGET_NO_PURCHASE + 0.02:
        nudge = "You are above target — lean toward purchase for borderline queries."
    else:
        nudge = "You are on target — maintain current balance."
    return GPT_SYSTEM_TEMPLATE.format(n_np=n_no_purchase, n_seen=n_seen, rate_pct=rate*100, nudge=nudge)

### Sequential API call loop with running calibration

In [65]:
def run_gpt_simulation(query_df, batch_size=GPT_BATCH_SIZE, verbose=True):
    """Sequential batched API calls with running no-purchase calibration."""
    query_ids = sorted(query_df["srch_id"].unique())
    n_np = 0
    n_seen = 0
    results = []

    for batch_start in range(0, len(query_ids), batch_size):
        batch_ids = query_ids[batch_start : batch_start + batch_size]
        batch_texts = [format_query(query_df[query_df["srch_id"] == q]) for q in batch_ids]
        user_msg = "\n\n---\n\n".join(batch_texts)

        system_msg = build_system_prompt(n_np, n_seen)

        resp = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg},
            ],
            response_format={"type": "json_object"},
            temperature=0.5,
        )
        parsed = _json.loads(resp.choices[0].message.content)
        decisions = parsed.get("decisions", [])

        for d in decisions:
            results.append(d)
            n_seen += 1
            if d.get("choice") is None:
                n_np += 1

        if verbose:
            rate = n_np / n_seen if n_seen else 0.0
            print(f"Batch {batch_start//batch_size + 1}: processed {n_seen}/{len(query_ids)}, no_purchase rate {rate:.1%}")

    return pd.DataFrame(results)

### Part A — Simulated Choices and MNL Re-estimation

Run GPT-4o mini against the full dataset, then re-estimate the MNL using AI-generated bookings instead of human bookings.

In [66]:
# WARNING: this calls the OpenAI API and costs ~$1.50.
# Comment out and load from cache if you've already run it.

ai_decisions_df = run_gpt_simulation(problem7_df, batch_size=GPT_BATCH_SIZE)
ai_decisions_df.to_csv("problem7_ai_decisions.csv", index=False)

ai_np_rate = ai_decisions_df["choice"].isna().mean()
print(f"\nFinal AI no_purchase rate: {ai_np_rate:.1%} (target 30.0%)")
ai_decisions_df.head()

Batch 1: processed 25/8354, no_purchase rate 40.0%
Batch 2: processed 50/8354, no_purchase rate 36.0%
Batch 3: processed 74/8354, no_purchase rate 39.2%
Batch 4: processed 97/8354, no_purchase rate 37.1%
Batch 5: processed 120/8354, no_purchase rate 38.3%
Batch 6: processed 144/8354, no_purchase rate 38.9%
Batch 7: processed 161/8354, no_purchase rate 37.3%
Batch 8: processed 186/8354, no_purchase rate 36.0%
Batch 9: processed 211/8354, no_purchase rate 34.1%
Batch 10: processed 235/8354, no_purchase rate 32.8%
Batch 11: processed 260/8354, no_purchase rate 32.3%
Batch 12: processed 285/8354, no_purchase rate 32.6%
Batch 13: processed 309/8354, no_purchase rate 33.0%
Batch 14: processed 334/8354, no_purchase rate 32.9%
Batch 15: processed 357/8354, no_purchase rate 32.5%
Batch 16: processed 381/8354, no_purchase rate 32.0%
Batch 17: processed 406/8354, no_purchase rate 31.3%
Batch 18: processed 430/8354, no_purchase rate 31.6%
Batch 19: processed 455/8354, no_purchase rate 32.7%
Batch 

,srch_id,choice,reasoning
0,1,6.000000,The hotel has a decent review score and a comp...
1,75,27.000000,This hotel offers a good balance of price and ...
2,101,NaN,None of the options stand out significantly in...
3,218,94.000000,This hotel has a high review score and a reaso...
4,236,96.000000,This hotel has a high review score and a promo...


In [67]:
# Build a synthetic booking_bool column from AI decisions
ai_booked = problem7_df[["srch_id", "row_index"]].copy()
ai_booked["ai_booking_bool"] = 0
for _, dec in ai_decisions_df.iterrows():
    if pd.notna(dec["choice"]):
        ai_booked.loc[ai_booked["row_index"] == int(dec["choice"]), "ai_booking_bool"] = 1

ai_df = problem7_df.merge(ai_booked[["row_index", "ai_booking_bool"]], on="row_index", how="left")

# Re-fit MNL using the same estimate_mnl from P4, swapping in AI bookings
beta_ai, result_ai = estimate_mnl(ai_df, "AI re-estimation", target_col="ai_booking_bool")

comparison = pd.DataFrame({
    "parameter": PARAM_NAMES,
    "beta_human (P1)": beta_hat,
    "beta_ai (P7)": beta_ai,
    "difference": beta_ai - beta_hat,
    "abs_difference": np.abs(beta_ai - beta_hat),
})
comparison.sort_values("abs_difference", ascending=False)

/var/folders/hq/zkz6grkn2cd8t2w965gsrqr80000gn/T/ipykernel_74149/956666876.py:17: RuntimeWarning: overflow encountered in exp
  sum_exp_by_group = np.bincount(group_index, weights=np.exp(u))
/Users/akhilfernando-bell/.local/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


,parameter,beta_human (P1),beta_ai (P7),difference,abs_difference
0,intercept,-2.830533,-0.413678,2.416855,2.416855
5,prop_accesibility_score,0.552096,0.020551,-0.531546,0.531546
6,prop_log_historical_price,-0.036687,-0.506359,-0.469672,0.469672
4,prop_location_score,0.017333,-0.269038,-0.286372,0.286372
1,prop_starrating,0.477118,0.210175,-0.266943,0.266943
3,prop_brand_bool,0.234592,-0.013609,-0.248201,0.248201
8,promotion_flag,0.448787,0.623362,0.174575,0.174575
2,prop_review_score,0.121425,0.248595,0.127170,0.127170
7,price_usd,-0.007342,-0.003696,0.003647,0.003647


### Part B — Held-Out Prediction Comparison

Give GPT-4o mini a few-shot context of training queries with their actual booking outcomes, then ask it to predict the held-out test queries. Compare its accuracy to MNL's predictions on the same test set.

In [68]:
# Use the train/test split and examples_df from the setup cells above
test_problem7_df = problem7_df[problem7_df["srch_id"].isin(test_query_ids)].copy()

# Add a few-shot examples block to the prompt context
examples_text = "\n".join(
    f"- srch_id={int(r['srch_id'])}: actual_choice={'no_purchase' if pd.isna(r.get('actual_option_id')) else int(r['actual_option_id'])}"
    for _, r in examples_df.iterrows()
)
GPT_SYSTEM_TEMPLATE_PREDICT = GPT_SYSTEM_TEMPLATE + f"\n\nFor calibration, here are real customer choices from training data:\n{examples_text}\n"

# Run prediction with the augmented prompt
_orig_template = GPT_SYSTEM_TEMPLATE
globals()["GPT_SYSTEM_TEMPLATE"] = GPT_SYSTEM_TEMPLATE_PREDICT
ai_predictions_df = run_gpt_simulation(test_problem7_df, batch_size=GPT_BATCH_SIZE)
globals()["GPT_SYSTEM_TEMPLATE"] = _orig_template

ai_predictions_df.to_csv("problem7_ai_predictions.csv", index=False)
ai_predictions_df.head()

Batch 1: processed 25/1671, no_purchase rate 44.0%
Batch 2: processed 50/1671, no_purchase rate 28.0%
Batch 3: processed 75/1671, no_purchase rate 29.3%
Batch 4: processed 100/1671, no_purchase rate 29.0%
Batch 5: processed 123/1671, no_purchase rate 29.3%
Batch 6: processed 148/1671, no_purchase rate 28.4%
Batch 7: processed 172/1671, no_purchase rate 29.1%
Batch 8: processed 197/1671, no_purchase rate 28.9%
Batch 9: processed 222/1671, no_purchase rate 29.7%
Batch 10: processed 247/1671, no_purchase rate 32.8%
Batch 11: processed 272/1671, no_purchase rate 33.1%
Batch 12: processed 297/1671, no_purchase rate 33.7%
Batch 13: processed 322/1671, no_purchase rate 32.9%
Batch 14: processed 347/1671, no_purchase rate 32.0%
Batch 15: processed 372/1671, no_purchase rate 31.5%
Batch 16: processed 396/1671, no_purchase rate 31.6%
Batch 17: processed 421/1671, no_purchase rate 33.3%
Batch 18: processed 446/1671, no_purchase rate 32.7%
Batch 19: processed 471/1671, no_purchase rate 33.1%
Batch

,srch_id,choice,reasoning
0,75,27.000000,This hotel has a good review score of 4 and a ...
1,218,91.000000,This hotel has a high review score of 5 and is...
2,1297,NaN,There are no standout options in terms of revi...
3,1857,200.000000,This hotel has a perfect review score of 5 and...
4,1893,NaN,The options available do not offer significant...


In [69]:
# Build human ground truth and MNL predictions on test set
test_truth = (
    test_problem7_df.groupby("srch_id")
    .apply(lambda g: g.loc[g["booking_bool"] == 1, "row_index"].iloc[0] if g["booking_bool"].sum() else None)
)
test_truth.name = "truth"

# MNL prediction: argmax of v_j across the assortment, OR null if all v_j too small
u_test = compute_utilities(beta_hat, test_problem7_df[FEATURES].to_numpy(dtype=float))
test_problem7_df = test_problem7_df.assign(_u=u_test)

def mnl_predict(g):
    # P(no_purchase) = 1 / (1 + sum exp(u))
    p_no = 1.0 / (1.0 + np.exp(g["_u"]).sum())
    if p_no > 0.5:
        return None
    return int(g.loc[g["_u"].idxmax(), "row_index"])

mnl_pred = test_problem7_df.groupby("srch_id").apply(mnl_predict)
mnl_pred.name = "mnl_pred"

ai_pred = ai_predictions_df.set_index("srch_id")["choice"]
ai_pred = ai_pred.where(ai_pred.notna(), None)
ai_pred.name = "ai_pred"

compare = pd.concat([test_truth, mnl_pred, ai_pred], axis=1)
compare["mnl_correct"] = compare["truth"].astype("object") == compare["mnl_pred"]
compare["ai_correct"] = compare["truth"].astype("object") == compare["ai_pred"]

print(f"Test queries: {len(compare):,}")
print(f"MNL accuracy: {compare['mnl_correct'].mean():.3f}")
print(f"AI  accuracy: {compare['ai_correct'].mean():.3f}")
compare.head()

Test queries: 1,671
MNL accuracy: 0.092
AI  accuracy: 0.080


/var/folders/hq/zkz6grkn2cd8t2w965gsrqr80000gn/T/ipykernel_74149/2631551385.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["booking_bool"] == 1, "row_index"].iloc[0] if g["booking_bool"].sum() else None)
/var/folders/hq/zkz6grkn2cd8t2w965gsrqr80000gn/T/ipykernel_74149/2631551385.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mnl_pred = test_problem7_df.groupby("srch_id").apply(mnl_

,truth,mnl_pred,ai_pred,mnl_correct,ai_correct
srch_id,,,,,
75,43.000000,32.000000,27.000000,False,False
218,91.000000,NaN,91.000000,False,True
1297,155.000000,167.000000,NaN,False,False
1857,203.000000,199.000000,200.000000,False,False
1893,225.000000,214.000000,NaN,False,False
